## **코드 흐름 요약**
이 코드는 앙상블(Ensemble)과 TTA(Test Time Augmentation)를 활용하여 모델의 성능을 극대화하는 과정을 담고 있다.

* 데이터 준비 및 전처리: glob으로 이미지를 로드하고 폴더명을 기반으로 라벨을 생성. 특히 이미지의 비율을 유지하기 위해 PadSquare라는 커스텀 변환을 사용하고, 데이터 불균형 해소를 위해 WeightedRandomSampler를 도입했다.

* 다양한 모델 학습: timm 라이브러리를 통해 4가지 서로 다른 아키텍처(Tiny ViT, EfficientNetV2-S/M, RegNetY)를 학습한다. 각 모델은 EarlyStopping과 ReduceLROnPlateau 스케줄러를 통해 최적의 시점에 저장된다.

* 혼합 정밀도 학습(AMP): GradScaler와 autocast를 사용하여 연산 속도를 높이고 메모리 사용량을 절감했다.

* 추론 및 앙상블: 학습된 4개 모델에 대해 각각 TTA(회전, 반전 등 7가지 변형)를 적용하여 확률값을 계산한다.

* Soft Voting: 4개 모델의 TTA 결과(확률값)를 모두 더해 최종 라벨을 결정(Argmax)하고 제출 파일을 생성한다.

## **새롭게 알게 된 내용 / 어려운 내용 / 배울 점**
* TTA (Test Time Augmentation)는 추론 시에도 데이터를 변형하여 여러 번 예측한 뒤 평균을 내는 기법이고 모델이 특정 각도나 방향에 치우치지 않고 강건한(Robust) 예측을 할 수 있게 돕는 다는 것을 새롭게 알게 되었다.

* Label Smoothing을 통해 CrossEntropyLoss에 label_smoothing=0.05를 적용하여 모델이 정답에 대해 지나치게 확신(Over-confidence)하지 않도록 유도, 일반화 성능을 높일 수 있다는 것을 배울 수 있었다.

* TTA 연산 로직 과정 중 apply_tta 함수에서 7가지 변환을 torch.cat으로 묶고, 추론 후 다시 view(7, B, -1).mean(dim=0)으로 평균을 내는 과정은 텐서의 차원(Dimension)에 대한 내용이 어려웠다.

주요 코드

In [ ]:
# 1. 커스텀 패딩: 이미지의 비율을 깨뜨리지 않고 정사각형으로 변환
image = cv2.copyMakeBorder(image, top, bottom, left, right, cv2.BORDER_CONSTANT, value=self.value)

In [ ]:
# 2. 가중치 샘플러: 클래스별 데이터 불균형을 해소하기 위한 전략
weights = [1.0 / class_counts[label] for label in train['rock_type']]
sampler = WeightedRandomSampler(weights, num_samples=len(weights), replacement=True)

In [ ]:
# 3. TTA 및 앙상블 핵심: 4개 모델의 예측 확률을 합산하여 Soft Voting 수행
weighted_probs = probs1 + probs2 + probs3 + probs4
final_preds = np.argmax(weighted_probs, axis=1)